In [1]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import mediapipe as mp
import cv2
import numpy as np

In [2]:
# Step 1: Base options with your model path
base_options = python.BaseOptions(
    model_asset_path="C:/Users/manya/fitness-app/models/pose_landmarker_heavy.task"
)

# Step 2: Configure PoseLandmarkerOptions
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=True,
    running_mode=vision.RunningMode.VIDEO  # use VIDEO for cv2 frames
)

# Step 3: Create the landmarker
pose_landmarker = vision.PoseLandmarker.create_from_options(options)


In [3]:




# Helper function
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    return angle if angle <= 180 else 360-angle

# Webcam loop
cap = cv2.VideoCapture(0)
counter, stage = 0, None
timestamp = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Convert frame
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)

    # Use detect_for_video with timestamp
    result = pose_landmarker.detect_for_video(mp_image, timestamp)
    timestamp += 33  # ~30 FPS

    if result.pose_landmarks:
        landmarks = result.pose_landmarks[0]  # first person detected
        h, w, _ = frame.shape

    # Use indices instead of mp.PoseLandmark
        shoulder = [landmarks[11].x * w, landmarks[11].y * h]  # LEFT_SHOULDER
        elbow    = [landmarks[13].x * w, landmarks[13].y * h]  # LEFT_ELBOW
        wrist    = [landmarks[15].x * w, landmarks[15].y * h]  # LEFT_WRIST

        angle = calculate_angle(shoulder, elbow, wrist)


        # Curl counter logic
        if angle > 160:
            stage = "down"
        if angle < 30 and stage == "down":
            stage = "up"
            counter += 1
            print(counter)

        # Draw angle + counter
        cv2.putText(frame, str(int(angle)), tuple(np.multiply(elbow, [1, 1]).astype(int)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 2, cv2.LINE_AA)
        cv2.rectangle(frame, (0,0), (225,73), (245,117,16), -1)
        cv2.putText(frame, f"REPS: {counter}", (10,160),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (0,0,255), 2, cv2.LINE_AA)
        cv2.putText(frame, f"STAGE: {stage}", (60,60),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (0,0,255), 2, cv2.LINE_AA)

    cv2.imshow("Pose Estimation", frame)
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


1
2
3
4
5
6
7
8
9
10
11
